In [2]:
import cv2
import mediapipe as mp
import numpy as np
import time
# from playsound import playsound
import pygame
pygame.mixer.init()

# -----------------------------
# Initialize MediaPipe FaceMesh
# -----------------------------
mp_face = mp.solutions.face_mesh
face_mesh = mp_face.FaceMesh()

# Eye landmarks
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

# -----------------------------
# Variables
# -----------------------------
blink_count = 0
eye_closed_frames = 0
drowsy_frames = 0
attention_score = 100

focus_time = 0
distraction_time = 0

start_time = time.time()

# -----------------------------
# EAR Calculation Function
# -----------------------------
def calculate_ear(eye_points, landmarks, width, height):

    points = []

    for p in eye_points:
        x = int(landmarks[p].x * width)
        y = int(landmarks[p].y * height)
        points.append((x, y))

    vertical1 = np.linalg.norm(np.array(points[1]) - np.array(points[5]))
    vertical2 = np.linalg.norm(np.array(points[2]) - np.array(points[4]))
    horizontal = np.linalg.norm(np.array(points[0]) - np.array(points[3]))

    ear = (vertical1 + vertical2) / (2 * horizontal)

    return ear


# -----------------------------
# Start Webcam
# -----------------------------
cap = cv2.VideoCapture(0)

while True:

    success, frame = cap.read()
    if not success:
        break

    frame = cv2.flip(frame, 1)

    height, width, _ = frame.shape

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    result = face_mesh.process(rgb_frame)

    if result.multi_face_landmarks:

        for face in result.multi_face_landmarks:

            landmarks = face.landmark

            # -----------------------------
            # EAR Calculation
            # -----------------------------
            left_ear = calculate_ear(LEFT_EYE, landmarks, width, height)
            right_ear = calculate_ear(RIGHT_EYE, landmarks, width, height)

            average_ear = (left_ear + right_ear) / 2

            # -----------------------------
            # Blink Detection
            # -----------------------------
            if average_ear < 0.25:

                eye_closed_frames += 1
                drowsy_frames += 1

            else:

                if eye_closed_frames > 2:
                    blink_count += 1

                eye_closed_frames = 0
                drowsy_frames = 0

            # -----------------------------
            # Drowsiness Detection
            # -----------------------------
            if drowsy_frames == 15:

                cv2.putText(frame, "DROWSY!", (50,80),
                            cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,255),3)

                # playsound("alarm.mp3")
                pygame.mixer.music.load("alarm.mp3")
                pygame.mixer.music.play()
            

                attention_score -= 2

            # -----------------------------
            # Head Direction Detection
            # -----------------------------
            nose_position = landmarks[1].x

            if nose_position < 0.4:

                cv2.putText(frame,"Looking Left",(50,130),
                            cv2.FONT_HERSHEY_SIMPLEX,1,(255,0,0),2)

                attention_score -= 1
                distraction_time += 1

            elif nose_position > 0.6:

                cv2.putText(frame,"Looking Right",(50,130),
                            cv2.FONT_HERSHEY_SIMPLEX,1,(255,0,0),2)

                attention_score -= 1
                distraction_time += 1

            else:

                focus_time += 1

            # -----------------------------
            # Display Blink Count
            # -----------------------------
            cv2.putText(frame,f"Blinks: {blink_count}",
                        (50,180),
                        cv2.FONT_HERSHEY_SIMPLEX,1,(0,255,0),2)

            # -----------------------------
            # Display Attention Score
            # -----------------------------
            cv2.putText(frame,f"Focus Score: {attention_score}",
                        (50,220),
                        cv2.FONT_HERSHEY_SIMPLEX,1,(255,255,0),2)

            # -----------------------------
            # Display Focus Time
            # -----------------------------
            cv2.putText(frame,f"Focus Time: {focus_time}",
                        (50,260),
                        cv2.FONT_HERSHEY_SIMPLEX,1,(0,255,255),2)

            # -----------------------------
            # Display Distraction Time
            # -----------------------------
            cv2.putText(frame,f"Distraction Time: {distraction_time}",
                        (50,300),
                        cv2.FONT_HERSHEY_SIMPLEX,1,(0,165,255),2)

    # -----------------------------
    # Show Window
    # -----------------------------
    cv2.imshow("Smart Study Monitor", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


# -----------------------------
# Release Camera
# -----------------------------
cap.release()
cv2.destroyAllWindows()